# ShopPulse: E-Commerce Sales Analytics Platform
## Notebook 01: Data Understanding & Initial Profiling

### 1. Project Background & Context
**ShopPulse** is a multi-category omnichannel e-commerce retail platform. The executive team requires an end-to-end analytical assessment of sales performance, profitability, customer acquisition/retention, product margins, and regional variance across historical transactions.

### 2. Objectives of this Notebook:
- Ingest raw transactional data (`data/raw/raw_ecommerce_data.csv`)
- Inspect structural metadata (schema, datatypes, shape, memory footprint)
- Profile missing values, null patterns, and duplicate records
- Analyze statistical distributions of transactional metrics (`sales`, `quantity`, `discount`, `profit`)
- Document data quality issues to address in the data cleaning pipeline


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Plot styling configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("Libraries imported successfully.")


### 3. Load Raw Transactional Dataset


In [ ]:
raw_data_path = '../data/raw/raw_ecommerce_data.csv'
df_raw = pd.read_csv(raw_data_path)

print(f"Raw Dataset Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
df_raw.head(5)


### 4. Data Types and Schema Inspection


In [ ]:
print("--- Data Schema & Non-Null Counts ---")
df_raw.info()


### 5. Missing Value Profiling & Nullity Audit


In [ ]:
null_counts = df_raw.isnull().sum()
null_pct = (null_counts / len(df_raw)) * 100
missing_df = pd.DataFrame({
    'Missing Count': null_counts,
    'Percentage (%)': null_pct.round(2)
}).sort_values(by='Missing Count', ascending=False)

print("--- Missing Values Summary ---")
print(missing_df[missing_df['Missing Count'] > 0])

# Visualize Missing Values
plt.figure(figsize=(10, 4))
missing_df[missing_df['Missing Count'] > 0]['Missing Count'].plot(kind='bar', color='#e74c3c')
plt.title('Missing Value Count by Feature', fontsize=14, fontweight='bold', pad=15)
plt.ylabel('Null Record Count')
plt.xlabel('Features')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### 6. Duplicate Record Detection


In [ ]:
exact_duplicates = df_raw.duplicated().sum()
order_id_duplicates = df_raw.duplicated(subset=['order_id']).sum()

print(f"Exact Duplicate Rows: {exact_duplicates:,} ({(exact_duplicates/len(df_raw)*100):.2f}%)")
print(f"Duplicate Order IDs: {order_id_duplicates:,} ({(order_id_duplicates/len(df_raw)*100):.2f}%)")


### 7. Numerical Summary & Distribution Analysis


In [ ]:
numerical_cols = ['quantity', 'unit_price', 'discount', 'sales', 'cost', 'profit']
summary_stats = df_raw[numerical_cols].describe().T
summary_stats['median'] = df_raw[numerical_cols].median()
summary_stats[['mean', 'std', 'min', '25%', 'median', '75%', 'max']]


### 8. Categorical Cardinality & Unique Entities


In [ ]:
cat_cols = ['category', 'region', 'city', 'payment_method', 'customer_segment']
for col in cat_cols:
    n_unique = df_raw[col].nunique()
    print(f"Column '{col}': {n_unique} unique values -> {list(df_raw[col].dropna().unique())[:8]}")


### 9. Key Findings & Cleaning Action Items
1. **Duplicates**: 150 duplicate rows detected that must be pruned.
2. **Missing Customer Names**: ~100 records contain missing customer names, which can be imputed via `customer_id` mapping.
3. **Missing Payment Methods**: Impute missing payment channels using customer default or mode.
4. **Date Formatting**: `order_date` contains mixed string representations and should be parsed to standard datetime format and calendar features extracted.
5. **Whitespace Inconsistencies**: Category and regional strings contain leading/trailing whitespaces requiring trimming.
6. **Financial Coherence**: Validate that `sales = quantity * price * (1 - discount)` and `profit = sales - cost` across all transactions.
